In [ ]:
%pip install pretty_midi torch numpy matplotlib scikit-learn

## 1. Load MIDI files
Recursively search data folder and return all MIDI files from the specified composers.

In [2]:
from pathlib import Path
import pretty_midi
import numpy as np

PROJECT_ROOT = Path("..")
DATA_ROOT = PROJECT_ROOT / "data" / "classic-midi-files-raw"

COMPOSERS = ["Beethoven", "Bach"]

midi_files = []
for comp in COMPOSERS:
    comp_dir = DATA_ROOT / comp
    midi_files.extend(sorted(comp_dir.rglob("*.[mM][iI][dD]"))) 

print(f"Found {len(midi_files)} MIDI files total from {COMPOSERS}.")
print("First few files:")
for p in midi_files[:5]:
    print(" ", p)
                      

Found 1237 MIDI files total from ['Beethoven', 'Bach'].
First few files:
  ../data/classic-midi-files-raw/Beethoven/32 Variations on a theme.mid
  ../data/classic-midi-files-raw/Beethoven/Andante in F Major.mid
  ../data/classic-midi-files-raw/Beethoven/Anh06 Rondo.mid
  ../data/classic-midi-files-raw/Beethoven/Anh08Nb1 Gavotte 4 hands.mid
  ../data/classic-midi-files-raw/Beethoven/Anhang 14-3.mid


## 2. Preprocessing and Tokenization
Melodies are extracted from each MIDI file, notes converted into `(pitch, duration_bucket)` tokens, and unusable or too‑short files skipped. Each note is represented as a pair of pitch and duration bucket, which allows the model to capture both melody and simplified rhythmic information. Durations are grouped into four categories (very short, short, medium, long) to reduce variability in raw MIDI timing and keep the vocabulary manageable. This representation enables the model to generate sequences that sound more musical than using pitch alone.



In [ ]:
# All functions in this code cell were AI-generated via ChatGPT

def choose_melody_instrument(midi: pretty_midi.PrettyMIDI):
    """Return a single pretty_midi.Instrument to treat as the melody, or None."""
    # Filter out drums
    candidates = [inst for inst in midi.instruments if not inst.is_drum]
    if not candidates:
        return None
    
    # Prefer Acoustic Grand Piano (program 0), but fall back to all non-drum
    piano_candidates = [inst for inst in candidates if inst.program == 0]
    if piano_candidates:
        candidates = piano_candidates
    
    # If instrument has no notes, skip it
    candidates = [inst for inst in candidates if inst.notes]
    if not candidates:
        return None
    
    # Pick instrument with highest average pitch (melody heuristic)
    def avg_pitch(inst):
        return np.mean([n.pitch for n in inst.notes])
    
    melody_inst = max(candidates, key=avg_pitch)
    return melody_inst


def duration_to_bucket(duration: float) -> int:
    """
    Map a duration in seconds to a small number of buckets.
    """
    if duration < 0.2:
        return 0  # very short (eighth-ish)
    elif duration < 0.5:
        return 1  # short (quarter-ish)
    elif duration < 1.0:
        return 2  # medium (half-ish)
    else:
        return 3  # long (whole+)


def notes_to_tokens(notes):
    """
    Convert a list of pretty_midi.Note objects into (pitch, duration_bucket) tokens,
    sorted by start time.
    """
    # Sort notes by start time
    notes = sorted(notes, key=lambda n: n.start)
    
    tokens = []
    for note in notes:
        pitch = note.pitch
        duration = note.end - note.start
        bucket = duration_to_bucket(duration)
        tokens.append((pitch, bucket))
    return tokens


In [13]:
# Build token sequences from all discovered MIDI files
import warnings
warnings.filterwarnings(
    "ignore",
    message="Tempo, Key or Time signature change events found",
    category=RuntimeWarning
)

SEQ_LEN = 50  # context window length

all_token_seqs = []      # list of lists of (pitch, bucket)
seq_file_paths = []      # parallel list of file paths (as strings)

num_loaded = 0
num_skipped_too_short = 0
num_skipped_no_melody = 0
num_failed_load = 0

for midi_path in midi_files:
    try:
        midi = pretty_midi.PrettyMIDI(str(midi_path))
    except Exception:
        num_failed_load += 1
        continue

    melody = choose_melody_instrument(midi)
    if melody is None:
        num_skipped_no_melody += 1
        continue

    tokens = notes_to_tokens(melody.notes)

    if len(tokens) < SEQ_LEN + 1:
        num_skipped_too_short += 1
        continue

    all_token_seqs.append(tokens)
    seq_file_paths.append(str(midi_path))
    num_loaded += 1


# Summary
print("\n=== Summary ===")
print("Total files found:        ", len(midi_files))
print("Successfully loaded:      ", num_loaded)
print("Skipped (no melody):      ", num_skipped_no_melody)
print("Skipped (too short):      ", num_skipped_too_short)
print("Failed to load:           ", num_failed_load)
print("Usable token sequences:   ", len(all_token_seqs))


=== Summary ===
Total files found:         1237
Successfully loaded:       1079
Skipped (no melody):       0
Skipped (too short):       157
Failed to load:            1
Usable token sequences:    1079


## 3. Dataset construction (X, y)
This section converts token sequences into numerical IDs, builds the vocabulary, and constructs input sequences (X) and next-token targets (y) for training.

In [ ]:
# AI-generated via ChatGPT 
# Build vocabulary from all (pitch, duration_bucket) tokens

token_to_id = {}
for seq in all_token_seqs:
    for tok in seq:
        if tok not in token_to_id:
            token_to_id[tok] = len(token_to_id)

id_to_token = {i: tok for tok, i in token_to_id.items()}

print("Vocab size:", len(token_to_id))

Vocab size: 307


In [ ]:
# AI-generated via ChatGPT 
# Build X (contexts), y (next-token targets), and example_file_indices

X_ids = []
y_ids = []
example_file_indices = [] 

for file_idx, seq in enumerate(all_token_seqs):
    # Convert (pitch, bucket) tokens to IDs
    seq_ids = [token_to_id[t] for t in seq]

    if len(seq_ids) <= SEQ_LEN:
        continue  

    # Slide a window of length SEQ_LEN
    for i in range(len(seq_ids) - SEQ_LEN):
        context = seq_ids[i : i + SEQ_LEN]
        target = seq_ids[i + SEQ_LEN]

        X_ids.append(context)
        y_ids.append(target)
        example_file_indices.append(file_idx)

X_ids = np.array(X_ids, dtype=np.int64)
y_ids = np.array(y_ids, dtype=np.int64)
example_file_indices = np.array(example_file_indices, dtype=np.int64)

print("X shape:", X_ids.shape)
print("y shape:", y_ids.shape)
print("Number of examples:", len(X_ids))
print("Unique source files used:", len(set(example_file_indices)))


X shape: (773541, 50)
y shape: (773541,)
Number of examples: 773541
Unique source files used: 1079


## 4. Train/Val/Test Split
Perform a file-level split to avoid data leakage, creating separate train/validation/test sets. Save the resulting splits for future use.

In [16]:
import numpy as np
from sklearn.model_selection import train_test_split

np.random.seed(42)

unique_files = np.unique(example_file_indices)
n_files = len(unique_files)
print("Total unique source files:", n_files)

# First split: train vs (val+test) – 70% / 30%
train_files, temp_files = train_test_split(
    unique_files,
    test_size=0.30,
    random_state=42,
    shuffle=True,
)

# Second split: val vs test – split temp 50/50 → 15% / 15% overall
val_files, test_files = train_test_split(
    temp_files,
    test_size=0.50,
    random_state=42,
    shuffle=True,
)

print("Train files:", len(train_files))
print("Val files:  ", len(val_files))
print("Test files: ", len(test_files))

# Build boolean masks over examples
train_mask = np.isin(example_file_indices, train_files)
val_mask   = np.isin(example_file_indices, val_files)
test_mask  = np.isin(example_file_indices, test_files)

# Convert masks to index arrays (positions in X_ids / y_ids)
train_idx = np.where(train_mask)[0]
val_idx   = np.where(val_mask)[0]
test_idx  = np.where(test_mask)[0]

print("\nExamples per split:")
print("  Train examples:", len(train_idx))
print("  Val examples:  ", len(val_idx))
print("  Test examples: ", len(test_idx))

total = len(train_idx) + len(val_idx) + len(test_idx)
print("\nTotal examples accounted for:", total, " (expected:", len(X_ids), ")")


Total unique source files: 1079
Train files: 755
Val files:   162
Test files:  162

Examples per split:
  Train examples: 544255
  Val examples:   100285
  Test examples:  129001

Total examples accounted for: 773541  (expected: 773541 )


## 5. Save Processed Dataset
Save X, y, vocabulary, and split indices into compressed `.npz` files for use in training and generation notebooks.

In [17]:
from pathlib import Path
import numpy as np

BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

out_path = PROCESSED_DIR / "full_sequences.npz"

vocab_array = np.array(list(token_to_id.keys()), dtype=np.int64)

np.savez(
    out_path,
    X=X_ids,
    y=y_ids,
    file_indices=example_file_indices,
    vocab=vocab_array,
    seq_len=SEQ_LEN,
)

print("Saved full dataset to:", out_path)

splits_path = PROCESSED_DIR / "splits_filelevel_indices.npz"

np.savez(
    splits_path,
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,
    train_files=train_files,
    val_files=val_files,
    test_files=test_files,
)

print("Saved split indices to:", splits_path)
print("  Train examples:", len(train_idx))
print("  Val examples:  ", len(val_idx))
print("  Test examples: ", len(test_idx))


Saved full dataset to: /Users/catherinembata/music-generation-project/data/processed/full_sequences.npz
Saved split indices to: /Users/catherinembata/music-generation-project/data/processed/splits_filelevel_indices.npz
  Train examples: 544255
  Val examples:   100285
  Test examples:  129001
